In [1]:
import pandas as pd
import os
import glob
import numpy as np

In [2]:
path = r'../data/final/tables/annotations/filled' # use your path
all_files = glob.glob(os.path.join(path, "*.xlsx"))

df_from_each_file = (pd.read_excel(f, skiprows = 3) for f in all_files)
df = pd.concat(df_from_each_file, ignore_index=True)

c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)
c:\Users\LENOVO\anaconda3\envs\bp_digitalization\Lib\site-packages\openpyxl\worksheet\_reader.py:329: UserWarning: Data Validation extension is not supported and will be removed
  warn(msg)


In [3]:
df.dropna(subset=['id'], inplace=True)

In [4]:
# Keeping only evaluation of existing rows 

df['id'] = df['id'].astype(str)
df['id_general'] = df['id'].str.replace(r'X_', '', regex=True)

In [5]:
df[df['Wert korrekt? (Ja/ Nein)'] == 'Nein']

,id,filename,met,value,Wert korrekt? (Ja/ Nein),Korrigierte Wert (falls nötig),Alternativbegriff/ Umschreibung,Standardisierte Fehlerquelle,Begründung,Wert korrekt? (Nein),Notiz,Alternativbegriff/Umschreibung,id_general
24,X_315,315_0.jpg,gfz_value,NaN,Nein,0.8,NaN,4: Begriff nicht gefunden,One value is in the text and one in the image,NaN,NaN,NaN,315
32,315,315_0.jpg,fok_value,NaN,Nein,1.5,Die Eingangsebene (Fertigfußboden im Erdgescho...,4: Begriff nicht gefunden,Description without key term,NaN,NaN,NaN,315
33,315,315_0.jpg,fok_unit,NaN,Nein,m,Die Eingangsebene (Fertigfußboden im Erdgescho...,4: Begriff nicht gefunden,Description without key term,NaN,NaN,NaN,315
46,X_316,316_1.jpg,grz_value,NaN,Nein,0.2,NaN,4: Begriff nicht gefunden,difficult table format,NaN,NaN,NaN,316
48,X_316,316_1.jpg,gfz_value,NaN,Nein,0.4,NaN,4: Begriff nicht gefunden,difficult table format,NaN,NaN,NaN,316
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5843,X_25757,25757_0.jpg,grz_value,NaN,Nein,0.4,NaN,4: Begriff nicht gefunden,NaN,NaN,NaN,NaN,25757
5844,25757,25757_0.jpg,grz_value,0.05,Nein,0.5,NaN,1: Wert falsch extrahiert,Wert überhaupt zu niedrig für GRZ,NaN,NaN,NaN,25757
5845,X_25757,25757_0.jpg,gfz_value,NaN,Nein,1.2,NaN,4: Begriff nicht gefunden,NaN,NaN,NaN,NaN,25757
5846,X_25757,25757_0.jpg,gfz_value,NaN,Nein,1.2,NaN,4: Begriff nicht gefunden,"für WA2, MI & SO",NaN,NaN,NaN,25757


In [6]:
df['correct_result'] = np.select(
        [
            df['Wert korrekt? (Ja/ Nein)'] == 'Ja',
            (df['Wert korrekt? (Ja/ Nein)'] == 'Nein') & (df['value'].isna()),
            (df['Wert korrekt? (Ja/ Nein)'] == 'Nein') & (df['value'].notna()),
            (df['Wert korrekt? (Ja/ Nein)'].isna()) & (df['value'].isna())
        ],
        [
            'True positive', 
            'False negative', 
            'False positive', 
            'True negative'
        ],
        default=None
    )

In [7]:
# keep rows with all met that contains grz or gfz
subset = df[df['met'].str.contains('grz|gfz', na=False, case=False, regex=True)]

In [8]:
subset['correct_result'].value_counts()

correct_result
True negative     968
False negative     70
True positive      42
False positive     11
Name: count, dtype: int64

In [9]:
def determine_result_per_group(group):
        has_ja = group['Wert korrekt? (Ja/ Nein)'].eq('Ja').any()
        has_nein = group['Wert korrekt? (Ja/ Nein)'].eq('Nein').any()
        all_na = group['Wert korrekt? (Ja/ Nein)'].isna().all()

        if has_ja and has_nein:
            return '3. Failed extraction: LLM failed to extract all of the values correctly'
        elif has_ja:
            return '1. Correct extraction: LLM extracted all values correctly'
        elif has_nein:
            return '4. Failed extraction: LLM only extracted some of the values correctly'
        elif all_na:
            return '2. Correct extraction: no values present in the document'
        


def check_correct_result(df, 
                         evaluation_level = 'document' # can be document or row
                         ):
    """
    Groups by 'id' and checks for the following conditions:
    - If at least one 'Ja' and at least one 'Nein' exists -> 'failed to extract all correct metrics'
    - If at least one 'Ja' exists -> 'extracted all correct metrics'
    - If all values are NaN -> 'no extracted metric'
    - Otherwise -> 'incorrect metric'

    Args:
    df (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'id' and 'correct_result'.
    """
        
    if evaluation_level == 'document':

        # Group by 'id' and apply the function
        df = df.groupby('id_general').apply(determine_result_per_group).reset_index(name='correct_result')

    elif evaluation_level == 'row':

        df['correct_result'] = np.select(
            [
                df['Wert korrekt? (Ja/ Nein)'] == 'Ja',
                (df['Wert korrekt? (Ja/ Nein)'] == 'Nein') & (df['value'].isna()),
                (df['Wert korrekt? (Ja/ Nein)'] == 'Nein') & (df['value'].notna()),
                (df['Wert korrekt? (Ja/ Nein)'].isna()) & (df['value'].isna())
            ],
            [
                'True positive', 
                'False negative', 
                'False positive', 
                'True negative'
            ],
            default=None
        )

    return df


In [10]:
def evaluate_llm_performance_on_data(df, evaluation_level = 'row'):

    metrics_evaluation = []

    for metric in df['met'].unique():

        keyword_subset = df[df['met'] == metric].copy()  

        keyword_subset['value_match'] = keyword_subset['Wert korrekt? (Ja/ Nein)'].apply(lambda x: 1 if x == 'Ja' or pd.isna(x) else 0)

        evaluation_results = check_correct_result(keyword_subset, evaluation_level = evaluation_level).value_counts('correct_result').reset_index()

        evaluation_results['metric'] = metric

        metrics_evaluation.append(evaluation_results)
    
    return pd.concat(metrics_evaluation, axis=0)
    

In [11]:
def compute_cumulative_percentages(evaluate_data):
    """
    Computes cumulative percentages for each metric based on the correct_result column.

    Args:
    evaluate_data (pd.DataFrame): The input DataFrame.

    Returns:
    pd.DataFrame: A DataFrame with 'metric', 'correct_result', 'count', 'percent', and 'cumulative_percent'.
    """
    # Step 1: Compute total counts per metric
    evaluate_data['metric_total'] = evaluate_data.groupby('metric')['count'].transform('sum')

    # Step 2: Compute percentage for each row
    evaluate_data['percent'] = evaluate_data['count'] / evaluate_data['metric_total'] * 100

    # Step 3: Sort by metric and correct_result (based on its order)
    # Extract the numeric prefix from correct_result for sorting
    evaluate_data['order'] = evaluate_data['correct_result'].str.extract(r'^(\d+)').astype(int)
    evaluate_data = evaluate_data.sort_values(by=['metric', 'order'])

    # Step 4: Compute cumulative sum of percentages within each metric
    evaluate_data['cumulative_percent'] = evaluate_data.groupby('metric')['percent'].cumsum().round(2)

    evaluate_data = evaluate_data.drop(columns=['metric_total', 'order'])

    return evaluate_data

def evaluate_and_format_llm_performance(data, 
                                        evaluation_level, 
                                        cummulative_percentages = True):
    
    if cummulative_percentages: 

        evaluation_results = evaluate_llm_performance_on_data(data, evaluation_level = evaluation_level)
        evaluation_results = compute_cumulative_percentages(evaluation_results)

        evaluation_results = evaluation_results.pivot(index='correct_result', columns='metric', values='cumulative_percent').fillna(0)

        return evaluation_results

    else:
    
        evaluation_results = evaluate_llm_performance_on_data(data, evaluation_level = evaluation_level)

        #evaluation_results = evaluation_results.pivot(index='correct_result', columns='metric', values='count').fillna(0)

        return evaluation_results

In [12]:
evaluate_data_per_row = evaluate_and_format_llm_performance(df, evaluation_level = 'row', cummulative_percentages = False)
#evaluate_data_per_row_cumulative = evaluate_and_format_llm_performance(df, evaluation_level = 'row', cummulative_percentages = True)

evaluate_data_per_document = evaluate_and_format_llm_performance(df, evaluation_level = 'document', cummulative_percentages = False)
evaluate_data_per_document_cumulative = evaluate_and_format_llm_performance(df, evaluation_level = 'document', cummulative_percentages =True)

C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10668\2040670413.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('id_general').apply(determine_result_per_group).reset_index(name='correct_result')
C:\Users\LENOVO\AppData\Local\Temp\ipykernel_10668\2040670413.py:37: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby('id_general').apply(determine_result_per_group).reset_index(name=

In [27]:
data = evaluate_data_per_document
# Normalize counts to sum to 100% per 'metric'
data['percent'] = data.groupby('metric')['count'].transform(lambda x: 100 * x / x.sum())

In [13]:
evaluate_data_per_row

,correct_result,count,metric
0,True negative,481,grz_value
1,False negative,37,grz_value
2,True positive,23,grz_value
3,False positive,8,grz_value
0,True negative,487,gfz_value
1,False negative,33,gfz_value
2,True positive,19,gfz_value
3,False positive,3,gfz_value
0,True negative,526,hw100_value
1,False negative,12,hw100_value


In [14]:
#compute f1 score per metric

def compute_f1_score(df, metric):
    """
    Computes the F1 score for a given metric based on the evaluation results.

    Args:
    df (pd.DataFrame): The input DataFrame.
    metric (str): The metric for which to compute the F1 score.

    Returns:
    float: The F1 score for the given metric.
    """
    # Extract relevant rows for the specified metric
    metric_data = df[df['metric'] == metric]

    # Extract counts for True Positives, False Positives, and False Negatives
    tp = metric_data[metric_data['correct_result'] == 'True positive']['count'].sum()
    fp = metric_data[metric_data['correct_result'] == 'False positive']['count'].sum()
    fn = metric_data[metric_data['correct_result'] == 'False negative']['count'].sum()

    # Compute precision and recall
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0

    # Compute F1 score
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    return f1_score



compute_f1_score(evaluate_data_per_row, 'grz_value')
compute_f1_score(evaluate_data_per_row, 'gfz_value')

np.float64(0.5135135135135135)

In [15]:
compute_f1_score(evaluate_data_per_row, 'gfz_value')


np.float64(0.5135135135135135)

In [29]:
evaluate_data_per_row

,correct_result,count,metric
0,True negative,481,grz_value
1,False negative,37,grz_value
2,True positive,23,grz_value
3,False positive,8,grz_value
0,True negative,487,gfz_value
1,False negative,33,gfz_value
2,True positive,19,gfz_value
3,False positive,3,gfz_value
0,True negative,526,hw100_value
1,False negative,12,hw100_value


In [13]:
evaluate_data_per_row.to_csv('../data/final/tables/annotations/evaluation_results_per_row.csv', index= False)

In [ ]:
evaluate_data_per_document.to_csv('../data/final/tables/annotations/evaluation_llm_performance_per_document.csv', index = False)